# 04 — Predict all test matches, validate TSVs and stage the submission

Select **Python (ml-challenge-er)** and run cells from top to bottom. **Run All includes full test inference** after the preflight checks and runtime limit below. No training is repeated.

The saved 03 model scored **0.923709 macro F0.5** on its classifier audit, with threshold **0.64** and best iteration **557**. This is not a test/leaderboard score. France is present in the test data but was absent from training.

The original retrieval rate implied roughly 15 days for the test set. Cached posting lists plus an exact native BM25 loop reduced the same 5,000-reference retrieval to **19.2 seconds with four workers**, versus 3,703 seconds, with identical candidates, routes, ranks and scores. This suggests roughly 1.8 hours for test retrieval alone, **not a guaranteed end-to-end time**. This notebook benchmarks test feature extraction and GPU prediction too.

Requirements already available on the development machine: the pinned conda environment, NVIDIA GPU access, and a **C99 compiler (`cc`/GCC)**. The tiny ranking helper's complete C source is in `src/rank_postings.c`; setup compiles it locally. It performs the same BM25 calculation and top-K selection, not a different matching algorithm. No external data or services are used.

All intermediate files stay under cache/. Final files are `submission/output/matching_results.tsv` and `candidate_pairs.tsv`. Candidate lists are **exactly** the records fed to the model; matches are their thresholded subsets.

In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "8")
import sys, json, time, shutil
from pathlib import Path
import polars as pl
import numpy as np
import xgboost as xgb
from IPython.display import display

ancestors = (Path.cwd(), *Path.cwd().parents)
SRC = next((p / rel for p in ancestors for rel in
            (".", "src", "code/business_entity_resolution/src", "submission/code/business_entity_resolution/src")
            if (p / rel / "predict_submission.py").is_file()), None)
assert SRC is not None, "Open this notebook from the project or extracted submission package."
SRC = SRC.resolve()
sys.path.insert(0, str(SRC))
from candidate_retrieval import cache_parts, build_index
from fast_retrieval import prepare_fast_index, compile_ranker
from train_matcher import TEXT_COLUMNS, file_digest, atomic_json
from predict_submission import (ModelScorer, TargetStore, RetrievalPool, score_batch,
    build_target_store, run_predictions, export_outputs, verify_fast_retrieval, package_submission)
from validate_streaming import validate_outputs
assert pl.__version__ == "1.44.2" and xgb.__version__ == "3.4.2" and np.__version__ == "2.4.6"
print("Python:", sys.executable)


Python: /home/aline/.conda/envs/ml-challenge-er/bin/python


In [2]:
# Paths are configurable for both the development workspace and extracted submission.
CACHE_ROOT = (Path(os.environ["ER_CACHE_ROOT"]).expanduser().resolve() if os.environ.get("ER_CACHE_ROOT") else
              next((p / "cache/normalized_v1" for p in ancestors
                    if (p / "cache/normalized_v1/test/test_source1/manifest.json").is_file()), None))
assert CACHE_ROOT is not None, "Set ER_CACHE_ROOT to notebook 01's complete cache."
CANDIDATE_ROOT = Path(os.environ.get("ER_CANDIDATE_ROOT", str(CACHE_ROOT.parent / "candidates_v1"))).expanduser().resolve()
TRAIN_ROOT = Path(os.environ.get("ER_TRAIN_ROOT", str(CACHE_ROOT.parent / "training_v1"))).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("ER_MODEL_ROOT", str(TRAIN_ROOT / "model"))).expanduser().resolve()
if not MODEL_DIR.is_dir() and (SRC.parent / "models").is_dir():
    MODEL_DIR = SRC.parent / "models"  # Reproduce using the model shipped in the final zip.
PREDICT_ROOT = Path(os.environ.get("ER_PREDICT_ROOT", str(CACHE_ROOT.parent / "prediction_v1"))).expanduser().resolve()
PREDICT_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_ROOT = SRC.parents[2]  # submission/ or the extracted zip root, containing code/ and output/.
OUTPUT_DIR = PACKAGE_ROOT / "output"
TEST_INDEX = PREDICT_ROOT / "test_search.sqlite"
TEST_FAST = PREDICT_ROOT / "test_fast"
TARGET_STORE = PREDICT_ROOT / "test_targets.sqlite"
RUN_DIR = PREDICT_ROOT / "predictions"
WORKERS = 4
ANCHORS_PER_BATCH = 250
PREFLIGHT_PER_COUNTRY = 200
MAX_ESTIMATED_HOURS = 8.0  # Resource budget; preflight stops rather than silently launching a multi-day run.
TEAM_NAME = ""            # Optional now; required to create the final named zip.
TEAM_MEMBERS = ""         # Comma-separated member names. TSV generation does not depend on these fields.

reference_parts, reference_manifest = cache_parts(CACHE_ROOT, "test", 1)
if os.environ.get("ER_DATA_ROOT"):
    DATA_ROOT = Path(os.environ["ER_DATA_ROOT"]).expanduser().resolve()
else:
    DATA_ROOT = Path(reference_manifest["signature"]["path"]).parents[1]
assert (DATA_ROOT / "test/test_source1.tsv").is_file(), "Set ER_DATA_ROOT to the supplied dataset/ folder."
for source in (2,3):
    cache_parts(CACHE_ROOT, "test", source)
assert shutil.disk_usage(PREDICT_ROOT).free > 12 * 2**30, "Keep at least 12 GiB free for indices, records, outputs and optional ZIP."
print("References:", f"{reference_manifest['total_rows']:,}")
print("Model:", MODEL_DIR)
print("Output:", OUTPUT_DIR)
print("Free disk GiB:", round(shutil.disk_usage(PREDICT_ROOT).free / 2**30, 1))
# These checks run before any large new index build.
compile_ranker(TEST_FAST / "native")
scorer = ModelScorer(MODEL_DIR, device="cuda")
print("Model loaded; GPU inference verified. Threshold:", scorer.threshold, "trees used:", scorer.end)


References: 1,732,544
Model: /home/aline/Projects/ML-Challenge/cache/training_v1/model
Output: /home/aline/Projects/ML-Challenge/submission/output
Free disk GiB: 63.3
Model loaded; GPU inference verified. Threshold: 0.64 trees used: 558


## Verify optimized retrieval against the saved pilot

This verifies **all 5,000 saved pilot anchors**, including exact candidate IDs, route bits, ranks and BM25 scores. It reads no labels to select candidates and does not change the model or threshold. Derived lookup tables are built once and reused.

An extracted final package may use the included parity report instead of rebuilding training artifacts. Its code/configuration fingerprints must still match. Changing the retriever requires a new parity check, not blind reuse of this report.

In [3]:
PARITY_PATH = MODEL_DIR / "retrieval_parity.json"
pilot_dir = CANDIDATE_ROOT / "pilot"
train_index = CANDIDATE_ROOT / "train_search.sqlite"
if train_index.exists() and (pilot_dir / "pilot_candidates.parquet").exists():
    train_fast = CANDIDATE_ROOT / "train_fast"
    prepare_fast_index(CACHE_ROOT, "train", train_index, train_fast, scorer.meta["retrieval_config"]["max_postings"])
    compile_ranker(train_fast / "native")
    parity_report = verify_fast_retrieval(CACHE_ROOT, train_index, train_fast, pilot_dir,
                                        PARITY_PATH, scorer.meta["retrieval_config"], WORKERS)
else:
    parity_report = json.loads(PARITY_PATH.read_text())
    fp = parity_report["fingerprint"]
    assert parity_report["status"] == "PASS" and parity_report["anchors"] >= 5000
    assert fp["fast_code_sha256"] == file_digest(SRC / "fast_retrieval.py")
    assert fp["ranker_code_sha256"] == file_digest(SRC / "rank_postings.c")
    assert fp["original_retrieval_sha256"] == file_digest(SRC / "candidate_retrieval.py")
    assert fp["retrieval_config"] == scorer.meta["retrieval_config"]
print(json.dumps(parity_report, indent=2))


{
  "status": "PASS",
  "anchors": 5000,
  "candidate_pairs": 240963,
  "max_score_error": 0.0,
  "seconds": 19.106659386998217,
  "workers": 4,
  "fingerprint": {
    "fast_code_sha256": "3a0a1c91358cd9748670bd5e53436c04336c72117e4777f7b199dba0813af9dd",
    "ranker_code_sha256": "7514b1d8528bb84e1264d30f1a7843ecb0428de8a1d24d97c65b8cfcb6d003ce",
    "original_retrieval_sha256": "1ae47240941f85db486b3e3b7f8b5226611cd41dc64c6405ecdebb0f13a31920",
    "reference_candidate_sha256": "dd89ae450e382858681451c3e17dfbd7dc7b6f9f7ad2443376b596e5e6de5c81",
    "retrieval_config": {
      "name_k": 20,
      "address_k": 20,
      "gram_k": 10,
      "joint_k": 10,
      "max_postings": 100000
    }
  }
}


## Build test retrieval and random-access feature stores

All countries, including France, are retained. These derived stores avoid rescanning ten million Parquet target records for each batch of references. Index, posting and target-store construction checkpoint completed units. A small interrupted dense-statistics/frequency build restarts safely.

The saved model's original retrieval settings are preserved. Changes here would require validation and potentially retraining; no country-specific threshold is invented for France.

In [4]:
build_index(CACHE_ROOT, "test", TEST_INDEX)
prepare_fast_index(CACHE_ROOT, "test", TEST_INDEX, TEST_FAST, scorer.meta["retrieval_config"]["max_postings"])
build_target_store(CACHE_ROOT, "test", TARGET_STORE)
print("All test stores are ready.")


test index: 100,000/9,969,589 | 89,857 rows/s | ETA 1.8 min
test index: 200,000/9,969,589 | 85,722 rows/s | ETA 1.9 min
test index: 300,000/9,969,589 | 83,877 rows/s | ETA 1.9 min
test index: 400,000/9,969,589 | 81,612 rows/s | ETA 2.0 min
test index: 500,000/9,969,589 | 82,067 rows/s | ETA 1.9 min
test index: 600,000/9,969,589 | 81,305 rows/s | ETA 1.9 min
test index: 700,000/9,969,589 | 80,384 rows/s | ETA 1.9 min
test index: 800,000/9,969,589 | 76,205 rows/s | ETA 2.0 min
test index: 900,000/9,969,589 | 72,965 rows/s | ETA 2.1 min
test index: 1,000,000/9,969,589 | 72,757 rows/s | ETA 2.1 min
test index: 1,100,000/9,969,589 | 73,844 rows/s | ETA 2.0 min
test index: 1,200,000/9,969,589 | 73,588 rows/s | ETA 2.0 min
test index: 1,300,000/9,969,589 | 73,835 rows/s | ETA 2.0 min
test index: 1,400,000/9,969,589 | 72,562 rows/s | ETA 2.0 min
test index: 1,500,000/9,969,589 | 71,351 rows/s | ETA 2.0 min
test index: 1,600,000/9,969,589 | 71,738 rows/s | ETA 1.9 min
test index: 1,700,000/9,96

## Test preflight and end-to-end runtime estimate

This samples up to 200 references **per country**, runs actual retrieval, feature extraction and GPU scoring, and weights the timings by each country's reference count. It does not estimate accuracy because test labels are unavailable.

If the estimate exceeds `MAX_ESTIMATED_HOURS`, this cell stops before full inference. Inspect the measured rate before changing that resource budget. Estimates can change with disk/cache behavior, other workloads and country-specific record complexity.

In [5]:
references = pl.scan_parquet(reference_parts)
country_counts = references.group_by("country_key").len().sort("country_key").collect(engine="streaming")
preflight_rows = []
pool = RetrievalPool(TEST_INDEX, TEST_FAST, scorer.meta["retrieval_config"], WORKERS)
store = TargetStore(TARGET_STORE)
try:
    for country, total in country_counts.iter_rows():
        sample = (references.select(TEXT_COLUMNS).filter((pl.col("country_key") == country) &
                  (pl.col("entity_id").hash(seed=888) % 100 == 0)).head(PREFLIGHT_PER_COUNTRY).collect(engine="streaming"))
        if sample.height == 0:
            sample = references.select(TEXT_COLUMNS).filter(pl.col("country_key") == country).head(PREFLIGHT_PER_COUNTRY).collect()
        started = time.perf_counter()
        predicted = score_batch(sample, pool, store, scorer)
        elapsed = time.perf_counter()-started
        preflight_rows.append({"country": country, "references": total, "sample_rows": sample.height,
            "seconds": elapsed, "estimated_hours": elapsed/sample.height*total/3600,
            "mean_candidates": predicted["candidate_entity_ids"].list.len().mean(),
            "mean_predicted_matches": predicted["matched_entity_ids"].list.len().mean()})
finally:
    pool.close();store.close()
display(pl.DataFrame(preflight_rows))
estimated_hours = sum(r["estimated_hours"] for r in preflight_rows)
atomic_json(PREDICT_ROOT / "preflight.json", {"countries": preflight_rows, "estimated_scoring_hours": estimated_hours,
                                             "workers": WORKERS, "note": "Timing only, not test accuracy; excludes final validation/ZIP."})
print(f"Estimated full scoring time: {estimated_hours:.2f} hours, plus output validation/packaging.")
assert estimated_hours <= MAX_ESTIMATED_HOURS, "Measured runtime exceeds the configured budget. Inspect the rate before raising MAX_ESTIMATED_HOURS."


country,references,sample_rows,seconds,estimated_hours,mean_candidates,mean_predicted_matches
str,i64,i64,f64,f64,f64,f64
"""france""",259452,200,2.686744,0.968168,49.07,3.355
"""india""",809986,200,1.180756,1.328327,47.91,2.875
"""us""",663106,200,1.045236,0.962642,46.4,2.955


Estimated full scoring time: 3.26 hours, plus output validation/packaging.


## Predict every test reference

This is the long-running cell. It writes one output part per 250 references by default. Interrupt and rerun to resume committed parts. Use one notebook writer per prediction directory. Changing model, features, ranker, threshold, reference inputs or batch size requires a new `ER_PREDICT_ROOT`.

The GPU scores each retrieved pair with exactly the saved 43 features and best-iteration range. References with no candidates still receive empty output rows. Candidate lists are saved **before** threshold filtering, with no later hidden candidate removal.

In [6]:
run_manifest = run_predictions(CACHE_ROOT, TEST_INDEX, TEST_FAST, TARGET_STORE, scorer,
                               RUN_DIR, batch_rows=ANCHORS_PER_BATCH, workers=WORKERS)
print("Full prediction complete:", run_manifest["total_rows"], "references")


Predicted 250/1,732,544 references | 158.5 rows/s | ETA 3.04 h
Predicted 500/1,732,544 references | 174.5 rows/s | ETA 2.76 h
Predicted 750/1,732,544 references | 180.8 rows/s | ETA 2.66 h
Predicted 1,000/1,732,544 references | 182.7 rows/s | ETA 2.63 h
Predicted 1,250/1,732,544 references | 182.9 rows/s | ETA 2.63 h
Predicted 1,500/1,732,544 references | 184.0 rows/s | ETA 2.61 h
Predicted 1,750/1,732,544 references | 184.8 rows/s | ETA 2.60 h
Predicted 2,000/1,732,544 references | 186.1 rows/s | ETA 2.58 h
Predicted 2,250/1,732,544 references | 186.4 rows/s | ETA 2.58 h
Predicted 2,500/1,732,544 references | 186.2 rows/s | ETA 2.58 h
Predicted 2,750/1,732,544 references | 186.6 rows/s | ETA 2.58 h
Predicted 3,000/1,732,544 references | 187.2 rows/s | ETA 2.57 h
Predicted 3,250/1,732,544 references | 188.1 rows/s | ETA 2.55 h
Predicted 3,500/1,732,544 references | 188.8 rows/s | ETA 2.54 h
Predicted 3,750/1,732,544 references | 189.2 rows/s | ETA 2.54 h
Predicted 4,000/1,732,544 refer

## Export and validate the required TSV files

The validator checks every candidate and matched ID against the **raw test files**, all required Source 1 rows, headers, list uniqueness, prefixes and the matched-subset rule. It uses a disk-backed ID lookup to avoid the organizer validator's large in-memory candidate sets. It also requires the source-order row alignment produced by this notebook.

Validation checks format and consistency, not prediction quality. Missing labels mean the test score remains unknown.

In [7]:
export_summary = export_outputs(RUN_DIR, OUTPUT_DIR)
validation_report = validate_outputs(OUTPUT_DIR / "matching_results.tsv", OUTPUT_DIR / "candidate_pairs.tsv",
                                     DATA_ROOT / "test", PREDICT_ROOT / "validation")
print(json.dumps(validation_report, indent=2))
assert validation_report["status"] == "PASS"
print("Leaderboard upload file:", OUTPUT_DIR / "matching_results.tsv")


Validation ID index: Source 1, 1,732,544 IDs
Validation ID index: Source 2, 4,887,273 IDs
Validation ID index: Source 3, 5,082,316 IDs
Validated 100,000 references
Validated 200,000 references
Validated 300,000 references
Validated 400,000 references
Validated 500,000 references
Validated 600,000 references
Validated 700,000 references
Validated 800,000 references
Validated 900,000 references
Validated 1,000,000 references
Validated 1,100,000 references
Validated 1,200,000 references
Validated 1,300,000 references
Validated 1,400,000 references
Validated 1,500,000 references
Validated 1,600,000 references
Validated 1,700,000 references
{
  "status": "PASS",
  "rows": 1732544,
  "candidate_pairs": 81961992,
  "matches": 5344318,
  "matching_sha256": "6ecdf1e2caf19f7a1d46180389ed52068fc99949cf08b06db1909b3855d27d65",
  "candidate_sha256": "cb32ecb6604ab70df4e6cb27917ef49a415b5df37c2e83586b8e736b2b3ee399",
  "checks": "Raw test ID existence, all Source 1 rows exactly once in source order,

## Stage the model and optionally create the final ZIP

Both output TSVs, all source (including the native ranking source), pinned environment, saved model, model metadata and methodology are included. Raw datasets and intermediate cache files are excluded.

If `TEAM_NAME` and `TEAM_MEMBERS` are blank, the notebook stages the artifacts but does not invent your identity or create a final named ZIP. Set those two fields and rerun this cell when ready. Inspect the methodology document before uploading the final archive.

The archive contains `output/`, `code/` and `Documentation_template.md` directly at its root, as required. It is created locally; nothing is uploaded automatically.

In [8]:
archive = package_submission(PACKAGE_ROOT, MODEL_DIR, validation_report, parity_report,
                             run_manifest, TEAM_NAME, TEAM_MEMBERS)
if archive:
    print("Final submission archive:", archive)
print("Final TSVs:", OUTPUT_DIR)


Validated TSVs and model artifacts are staged. Fill TEAM_NAME and TEAM_MEMBERS, then rerun this packaging cell to create the final zip.
Final TSVs: /home/aline/Projects/ML-Challenge/submission/output
